In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
from pathlib import Path

data_path = Path("../../data/egx")

files = list(data_path.glob("*"))

for file in files:
    print(file.name)

In [ ]:
stock = "ABUK"

stock_path = Path(f"../../data/egx/{stock}.csv")

stock_df = pd.read_csv(stock_path)

print("Stock:", stock)
print("Shape:", stock_df.shape)

display(stock_df.head())

In [ ]:
print(stock_df.columns.tolist())

In [ ]:
print(stock_df.info())

In [ ]:
stock_df.isnull().sum()

In [ ]:
stock_df["date"] = pd.to_datetime(stock_df["date"])

stock_df = stock_df.sort_values("date").reset_index(drop=True)

display(stock_df.head())
display(stock_df.tail())

In [ ]:
stock_df["return"] = stock_df["close"].pct_change()

In [ ]:
display(
    stock_df[["date", "close", "return"]].head(10)
)

In [ ]:
stock_df = stock_df.dropna(subset=["return"]).reset_index(drop=True)

print("Rows after removing missing returns:", len(stock_df))

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(stock_df["date"], stock_df["return"])

plt.title(f"{stock} Daily Returns")
plt.xlabel("Date")
plt.ylabel("Daily Return")
plt.grid(True)

plt.show()

In [ ]:
for lag in range(1, 6):
    stock_df[f"return_lag_{lag}"] = stock_df["return"].shift(lag)

In [ ]:
display(
    stock_df[
        [
            "date",
            "return",
            "return_lag_1",
            "return_lag_2",
            "return_lag_3",
            "return_lag_4",
            "return_lag_5"
        ]
    ].head(10)
)

In [ ]:
stock_df = stock_df.dropna().reset_index(drop=True)

print("Final dataset shape:", stock_df.shape)

In [ ]:
features = [
    "return_lag_1",
    "return_lag_2",
    "return_lag_3",
    "return_lag_4",
    "return_lag_5"
]

X = stock_df[features]
y = stock_df["return"]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
split_index = int(len(X) * 0.70)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining period:")
print(stock_df.iloc[:split_index]["date"].min())
print("to")
print(stock_df.iloc[:split_index]["date"].max())

print("\nTesting period:")
print(stock_df.iloc[split_index:]["date"].min())
print("to")
print(stock_df.iloc[split_index:]["date"].max())

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [ ]:
print("X_train:", X_train_scaled.shape)
print("X_test :", X_test_scaled.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [ ]:
model = MLPRegressor(
    hidden_layer_sizes=(32,),
    activation="relu",
    solver="adam",
    random_state=42,
    shuffle=False
)

In [ ]:
epochs = 100

train_losses = []
test_losses = []

for epoch in range(epochs):
    
    model.partial_fit(X_train_scaled, y_train)
    
    train_pred = model.predict(X_train_scaled)
    test_pred = model.predict(X_test_scaled)
    
    train_loss = mean_squared_error(y_train, train_pred)
    test_loss = mean_squared_error(y_test, test_pred)
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)

print("Training completed.")

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    range(1, epochs + 1),
    train_losses,
    label="Training Loss"
)

plt.plot(
    range(1, epochs + 1),
    test_losses,
    label="Testing Loss"
)

plt.title(f"{stock} - Training Loss vs Testing Loss")
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    stock_df.iloc[:split_index]["date"],
    y_train.values,
    label="Actual"
)

plt.plot(
    stock_df.iloc[:split_index]["date"],
    y_train_pred,
    label="Predicted"
)

plt.title(f"{stock} - Training Period: Actual vs Predicted Returns")
plt.xlabel("Date")
plt.ylabel("Daily Return")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    stock_df.iloc[split_index:]["date"],
    y_test.values,
    label="Actual"
)

plt.plot(
    stock_df.iloc[split_index:]["date"],
    y_test_pred,
    label="Predicted"
)

plt.title(f"{stock} - Testing Period: Actual vs Predicted Returns")
plt.xlabel("Date")
plt.ylabel("Daily Return")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

train_directional_accuracy = (
    np.sign(y_train.values) == np.sign(y_train_pred)
).mean()

test_directional_accuracy = (
    np.sign(y_test.values) == np.sign(y_test_pred)
).mean()

print("===== Training Performance =====")
print(f"MSE: {train_mse:.8f}")
print(f"MAE: {train_mae:.8f}")
print(f"R²: {train_r2:.4f}")
print(f"Directional Accuracy: {train_directional_accuracy:.2%}")

print("\n===== Testing Performance =====")
print(f"MSE: {test_mse:.8f}")
print(f"MAE: {test_mae:.8f}")
print(f"R²: {test_r2:.4f}")
print(f"Directional Accuracy: {test_directional_accuracy:.2%}")

In [ ]:
def run_experiment(hidden_layers, epochs):
    
    model = MLPRegressor(
        hidden_layer_sizes=hidden_layers,
        activation="relu",
        solver="adam",
        max_iter=1,
        random_state=42,
        shuffle=False
    )
    
    train_losses = []
    test_losses = []
    
    for epoch in range(epochs):
        
        model.partial_fit(X_train_scaled, y_train)
        
        train_pred = model.predict(X_train_scaled)
        test_pred = model.predict(X_test_scaled)
        
        train_loss = mean_squared_error(y_train, train_pred)
        test_loss = mean_squared_error(y_test, test_pred)
        
        train_losses.append(train_loss)
        test_losses.append(test_loss)
    
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    
    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)
    
    train_direction = (
        np.sign(y_train.values) == np.sign(train_pred)
    ).mean()
    
    test_direction = (
        np.sign(y_test.values) == np.sign(test_pred)
    ).mean()
    
    return {
        "Architecture": str(hidden_layers),
        "Epochs": epochs,
        "Train MSE": train_losses[-1],
        "Test MSE": test_losses[-1],
        "Train MAE": train_mae,
        "Test MAE": test_mae,
        "Train R2": train_r2,
        "Test R2": test_r2,
        "Train Directional Accuracy": train_direction,
        "Test Directional Accuracy": test_direction,
        "Train Losses": train_losses,
        "Test Losses": test_losses,
        "Model": model
    }

In [ ]:
baseline = run_experiment(
    hidden_layers=(32,),
    epochs=100
)

print("Baseline completed.")

In [ ]:
experiments = []

architectures = [
    (16,),
    (32,),
    (64,),
    (128,),
    (64, 32),
    (128, 64),
    (128, 64, 32)
]

for architecture in architectures:
    
    result = run_experiment(
        hidden_layers=architecture,
        epochs=100
    )
    
    experiments.append(result)
    
    print(f"Completed: {architecture}")

In [ ]:
results_df = pd.DataFrame(experiments)

results_df[
    [
        "Architecture",
        "Epochs",
        "Train MSE",
        "Test MSE",
        "Train MAE",
        "Test MAE",
        "Train R2",
        "Test R2",
        "Test Directional Accuracy"
    ]
]

In [ ]:
results_sorted = results_df.sort_values(
    "Test MSE"
).reset_index(drop=True)

display(
    results_sorted[
        [
            "Architecture",
            "Epochs",
            "Test MSE",
            "Test MAE",
            "Test R2",
            "Test Directional Accuracy"
        ]
    ]
)

In [ ]:
plt.figure(figsize=(12, 6))

plt.bar(
    results_df["Architecture"],
    results_df["Test MSE"]
)

plt.title(f"{stock} - Test MSE by Network Architecture")
plt.xlabel("Network Architecture")
plt.ylabel("Test MSE")
plt.xticks(rotation=45)

plt.show()

In [ ]:
epoch_experiments = []

epoch_values = [25, 50, 100, 200, 300]

for epochs in epoch_values:
    
    result = run_experiment(
        hidden_layers=(32,),
        epochs=epochs
    )
    
    epoch_experiments.append(result)
    
    print(f"Completed: {epochs} epochs")

In [ ]:
epoch_results_df = pd.DataFrame(epoch_experiments)

display(
    epoch_results_df[
        [
            "Architecture",
            "Epochs",
            "Train MSE",
            "Test MSE",
            "Train MAE",
            "Test MAE",
            "Test R2",
            "Test Directional Accuracy"
        ]
    ]
)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    epoch_results_df["Epochs"],
    epoch_results_df["Test MSE"],
    marker="o"
)

plt.title(f"{stock} - Test MSE vs Training Epochs")
plt.xlabel("Epochs")
plt.ylabel("Test MSE")
plt.grid(True)

plt.show()

In [ ]:
best_result = results_sorted.iloc[0]

print("Best Architecture:", best_result["Architecture"])
print("Best Test MSE:", best_result["Test MSE"])

In [ ]:
best_architecture = eval(best_result["Architecture"])

best_experiment = run_experiment(
    hidden_layers=best_architecture,
    epochs=100
)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    range(1, 101),
    best_experiment["Train Losses"],
    label="Training Loss"
)

plt.plot(
    range(1, 101),
    best_experiment["Test Losses"],
    label="Testing Loss"
)

plt.title(
    f"{stock} - Best Architecture: {best_architecture}"
)

plt.xlabel("Epoch")
plt.ylabel("MSE")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Number of test observations to display
n_points = 150

actual = y_test.values[:n_points]
predicted = y_test_pred[:n_points]

plt.figure(figsize=(14, 6))

plt.plot(
    range(n_points),
    actual,
    label="Actual",
    linewidth=2
)

plt.plot(
    range(n_points),
    predicted,
    label="MLP Prediction",
    linestyle="--",
    linewidth=2
)

plt.title(f"{stock} - Prediction vs Actual")
plt.xlabel("Test Observation")
plt.ylabel("Daily Return")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
n_points = 150

actual_train = y_train.values[-n_points:]
predicted_train = y_train_pred[-n_points:]

plt.figure(figsize=(14, 6))

plt.plot(
    range(n_points),
    actual_train,
    label="actual next value",
    linewidth=2
)

plt.plot(
    range(n_points),
    predicted_train,
    label="MLP prediction",
    linestyle="--",
    linewidth=2
)

plt.title("Training - Prediction vs Actual")
plt.xlabel("Training Observation")
plt.ylabel("Daily Return")

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
n_points = 150

actual = y_test.values[-n_points:]
predicted = y_test_pred[-n_points:]

plt.figure(figsize=(14, 6))

plt.plot(
    range(n_points),
    actual,
    label="actual next value",
    linewidth=2
)

plt.plot(
    range(n_points),
    predicted,
    label="MLP prediction",
    linestyle="--",
    linewidth=2
)

plt.title("Prediction vs Actual")
plt.xlabel("Test Observation")
plt.ylabel("Daily Return")

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()